# CPLE — Combined Pseudo-Labeling & Entropy (example)

This notebook contains a compact, runnable example of a simple semi-supervised pipeline we call *CPLE* (Combined Pseudo-Labeling + Entropy minimization).

It demonstrates:

- creating a synthetic classification problem with some labeled and unlabeled samples,
- training a baseline supervised classifier on labeled data only,
- applying an iterative pseudo-labeling loop that adds high-confidence unlabeled examples to the labeled set,
- comparing accuracies and visualizing results,

Use this as a starting point to compare with the S3VM/TSVM examples in the repository and to run experiments listed in the 
 section at the end.

## What is CPLE (informal)

CPLE here refers to a lightweight semi-supervised strategy that combines two ideas:

1. Pseudo-labeling: train a classifier on labeled data, predict labels for unlabeled points, then add *high-confidence* predictions to the labeled set and repeat.

2. Entropy minimization (soft): encourage confident predictions on unlabeled data (we implement this by preferring high-probability pseudo-labels).

Pros: easy to implement and often effective; Cons: can amplify label noise and create confirmation bias. Use thresholds and sanity checks when adding pseudo-labels.

In [ ]:
# Imports and dataset generation
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(1000)

# Create a small 2D classification problem (easy to visualize)
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0, n_clusters_per_class=1, class_sep=1.2, random_state=1000)
# Convert labels 0->-1 and 1 stays 1 for plotting parity with other scripts if needed
y_binary = y.copy()
y_binary[y_binary == 0] = -1

# Hold out a test set for final evaluation
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# From the trainval set, keep only a small fraction labeled; the rest are 'unlabeled' (we retain true labels for evaluation)
n_labeled = 20
indices = np.arange(X_trainval.shape[0])
np.random.shuffle(indices)
labeled_idx = indices[:n_labeled]
unlabeled_idx = indices[n_labeled:]

X_labeled = X_trainval[labeled_idx]
y_labeled = y_trainval[labeled_idx]
X_unlabeled = X_trainval[unlabeled_idx]
y_unlabeled_true = y_trainval[unlabeled_idx]  # kept for evaluation only

print('Train/val size:', X_trainval.shape[0], '   labeled:', X_labeled.shape[0], '  unlabeled:', X_unlabeled.shape[0])

In [ ]:
# Baseline: supervised classifier trained on labeled data only
clf = LogisticRegression(solver='lbfgs')
clf.fit(X_labeled, y_labeled)
y_pred_test = clf.predict(X_test)
acc_supervised = accuracy_score(y_test, y_pred_test)
print(f'Baseline supervised (labeled-only) accuracy on test set: {acc_supervised:.4f}')

# Helper: plot current labeled / unlabeled sets and decision boundary (2D only)
def plot_sets(clf=None, X_labeled=None, y_labeled=None, X_unlabeled=None, title=None):
    plt.figure(figsize=(6,5))
    if X_unlabeled is not None:
        plt.scatter(X_unlabeled[:,0], X_unlabeled[:,1], facecolors='none', edgecolors='gray', label='unlabeled')
    if X_labeled is not None:
        plt.scatter(X_labeled[y_labeled==0,0], X_labeled[y_labeled==0,1], c='C0', marker='o', label='labeled class 0')
        plt.scatter(X_labeled[y_labeled==1,0], X_labeled[y_labeled==1,1], c='C1', marker='^', label='labeled class 1')
    if clf is not None:
        # Decision boundary contour
        xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200), np.linspace(X[:,1].min()-1, X[:,1].max()+1, 200))
        Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)
        plt.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=1)
    plt.legend()
    plt.title(title)
    plt.show()

plot_sets(clf=clf, X_labeled=X_labeled, y_labeled=y_labeled, X_unlabeled=X_unlabeled, title='Baseline labeled set and decision boundary')

In [ ]:
# Pseudo-labeling loop (simple):
# - Train on current labeled set
# - Predict probabilities on unlabeled set
# - Add samples with prob >= threshold or <= (1-threshold) to labeled set with their pseudo-labels
# - Repeat for a few iterations or until no more additions

def pseudo_labeling(X_labeled, y_labeled, X_unlabeled, threshold=0.95, max_iter=10):
    X_l = X_labeled.copy()
    y_l = y_labeled.copy()
    X_u = X_unlabeled.copy()
    for it in range(max_iter):
        clf = LogisticRegression(solver='lbfgs', max_iter=500)
        clf.fit(X_l, y_l)
        probs = clf.predict_proba(X_u)
        high_conf_mask = (probs.max(axis=1) >= threshold)
        if not np.any(high_conf_mask):
            # nothing to add
            print(f'Iteration {it}: no high-confidence samples (threshold={threshold}) -- stopping')
            break
        add_X = X_u[high_conf_mask]
        add_y = clf.predict(X_u[high_conf_mask])
        # Append to labeled set
        X_l = np.vstack([X_l, add_X])
        y_l = np.hstack([y_l, add_y])
        # Remove added from unlabeled pool
        X_u = X_u[~high_conf_mask]
        print(f'Iteration {it}: added {add_X.shape[0]} samples, unlabeled remaining: {X_u.shape[0]}')
    # final classifier
    final_clf = LogisticRegression(solver='lbfgs', max_iter=500)
    final_clf.fit(X_l, y_l)
    return final_clf, X_l, y_l, X_u

# Run pseudo-labeling
clf_pl, X_l_pl, y_l_pl, X_u_remaining = pseudo_labeling(X_labeled, y_labeled, X_unlabeled, threshold=0.98, max_iter=10)
y_pred_test_pl = clf_pl.predict(X_test)
acc_pl = accuracy_score(y_test, y_pred_test_pl)
print(f'After pseudo-labeling accuracy on test set: {acc_pl:.4f}')
plot_sets(clf=clf_pl, X_labeled=X_l_pl, y_labeled=y_l_pl, X_unlabeled=X_u_remaining, title='After pseudo-labeling')

## Tasks / experiments to try

1. Compare this simple CPLE pseudo-labeling approach with the `s3vm.py` and `tsvm.py` scripts in the repo: measure test accuracy and label quality of pseudo-labels.
2. Vary the confidence threshold (e.g., 0.9, 0.95, 0.98) and the initial labeled set size; plot accuracy vs threshold and vs number of labeled samples.
3. Try a softer entropy-minimization: instead of hard adding only very confident points, add lower-confidence points with lower weight (e.g., via sample weighting) or add pseudo-labels but mark them as 'soft' in a loss that uses predicted probability.
4. Evaluate robustness to label noise: flip some fraction of initial labels and observe how pseudo-labeling amplifies or mitigates noise.
5. Replace LogisticRegression with other models (RandomForest, SVM, neural net). Compare speed and behavior.
6. Visualize decision boundary evolution across iterations (capture classifier each iteration).
7. Implement cross-validation to tune threshold and number of iterations to avoid overfitting.

If you want, I can also add a cell that runs a small grid search over thresholds and initial labeled sizes and saves results to CSV for plotting.

## How to run

Open the notebook and run all cells. The notebook uses only scikit-learn, numpy and matplotlib (and optionally seaborn). If you don't have them installed, install with pip: `pip install numpy scikit-learn matplotlib`.

Suggested next step: run the notebook, then try `threshold=0.95` and `threshold=0.9` to see the trade-offs.